# **1 INITIAL CHECK OF THE DATASET "Global data on sustainable energy (2000-2020)"**

## 1.1 Setup and loading


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Reading from a relative path (works for anyone who clones the repository)
df = pd.read_csv('global-data-on-sustainable-energy.csv')

df.head(10)

The **Density** variable (population density) appears to show the same value over time. In the following section, the values of this variable are checked across all rows to verify whether this behavior is consistent.

## 1.2 Dataset structure



In [ ]:
print('Rows, columns:', df.shape)

df.info()

**Note:** `Density\n(P/Km2)` has dtype **object** (text), not `float64` like the other variables, due to a formatting error (comma used as a thousands separator).

In [ ]:
rows_per_country = df['Entity'].value_counts()
print(rows_per_country.describe())

# The few countries with fewer than 21 years (consistent with geopolitics: Serbia/Montenegro,
# South Sudan, French Guiana)
print('Countries with fewer than 21 years:')
print(rows_per_country[rows_per_country < 21])

In [ ]:
# Check whether population density changes over the years, for each country
density_per_country = df.groupby('Entity')['Density\\n(P/Km2)'].nunique()
print('Countries with constant density across all years:', (density_per_country == 1).sum())
print('Countries with different density values between years:', (density_per_country > 1).sum())
print('Total countries in the dataset:', len(density_per_country))

Since population density is constant over time for almost all countries,
to make this variable more realistic, we **replace it** with an external dataset
that reports a different value for each year:

### 1.2.1 Structure of the external "Density" dataset

In [ ]:
density = pd.read_csv("density.csv", sep=';', decimal=',')

print('rows, columns:', density.shape)

density.info()

In [ ]:
# Descriptive statistics of the density values
print(density['Valore'].describe().map('{:,.2f}'.format))

**Note:** the maximum value (14,891,192) is extremely distant from the 75th percentile
(180.46), and the standard deviation exceeds the mean — indicating the presence of at least one
outlier that distorts the distribution.

In [ ]:
# Check which rows exceed a plausibility threshold
# (the highest population density in the world, Monaco, is around 26,000 inhabitants/Km2:
# we use 50,000 as a safety threshold, well above any realistic value)

suspicious_values = density[density['Valore'] > 50000]
print(suspicious_values[['Entity', 'Year', 'Valore']])

In [ ]:
# Correct the anomalous value: the decimal separator appears to be missing.
# 14891192 becomes 14.891192, consistent with the increasing trend in the following years
# (2001: 15.27, 2002: 15.65...)

density.loc[(density['Entity']=='Sudan') & (density['Year']==2000), 'Value'] = 14891192 / 1_000_000

print('Value after correction:')
print(density[(density['Entity']=='Sudan') & (density['Year']==2000)])

In [ ]:
# Confirm the effect of the correction on the descriptive statistics

print(density['Value'].describe().map('{:,.2f}'.format))

In [ ]:
# Density column replacement in the main dataset (Merge)

global_file = "global-data-on-sustainable-energy.csv"

df = pd.read_csv(global_file)

initial_length = len(df)
print("Initial length:", initial_length)

# Remove the old density column (it will be replaced by the new one)
df = df.drop(columns=['Density\\n(P/Km2)'])

# Keep only the columns needed from the density file (already corrected above)
clean_density = density[['Entity', 'Year', 'Value']].rename(columns={'Value': 'Density\\n(P/Km2)'})

# Actual merge: matches rows by Entity + Year, not by position
df = df.merge(clean_density, on=['Entity', 'Year'], how='left')

final_length = len(df)
print("Final length:", final_length)

if initial_length == final_length:
    print("OK: dataset length is unchanged")
else:
    print("ERROR: length has changed")

# Fundamental check: how many rows did NOT find a corresponding density value?
missing_after_merge = df['Density\\n(P/Km2)'].isna().sum()
print("Rows without density after merge:", missing_after_merge)

df.to_csv("Merged_global-data-on-sustainable-energy.csv", index=False)
print("File saved correctly")

In [ ]:
df = pd.read_csv('Merged_global-data-on-sustainable-energy.csv')

df.head(10)

In [ ]:
# Check the data type of the updated Density column: we expect float64, no longer object

print(df['Density\\n(P/Km2)'].dtype)

## 1.3 Missing values

For each column, we calculate the percentage of missing values. `.isna()` returns
a table of True/False values; `.mean()` on the column gives the fraction of True values (i.e.
missing values). We sort from the worst.

In [ ]:
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]

In [ ]:
# Display missing values as horizontal bars

m = missing[missing > 0]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(m.index[::-1], m.values[::-1], color='#2f7d8c')
ax.axvline(50, color='#d03b3b', linestyle='--', linewidth=1)
ax.text(51, 0.5, '50% threshold\n(columns to discard)', color='#d03b3b', fontsize=8)
ax.set_xlabel('% missing values')
ax.set_title('Missing values by column')
plt.tight_layout(); plt.show()

**Interpretation:** three categories. *Critical* (>50%: `Renewables % equivalent primary energy` 59%,
`Financial flows` 57%) -> columns to discard, imputing half of the column would mean inventing
data. *Medium* (10-26%: renewable capacity per capita, CO2) -> missing values are not random
(mostly missing for small/poor countries). *Low* (<9%) -> safe imputation using the
median.

In [ ]:
# Drop columns with more than 50% missing values
columns_to_drop = ['Renewables (% equivalent primary energy)', 'Financial flows to developing countries (US $)']
df = df.drop(columns=columns_to_drop)

print('Remaining columns:', df.shape[1])

## 1.4 Construction of the training set and test set

Split performed by country into train and test sets. This means that the same country does not appear in both the train and test sets. This helps prevent the model from "cheating" and avoids biasing the evaluation metrics on the test set later on.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

target_col = 'Renewable energy share in the total final energy consumption (%)'

df = df.dropna(subset=[target_col]).reset_index(drop=True)

# SPLIT BY COUNTRY: ~20% of countries (with ALL their years) goes into the test set.
# This way, no country is divided between train and test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['Entity']))
train_set = df.iloc[train_idx].copy()
test_set  = df.iloc[test_idx].copy()

print('Train set:', train_set.shape, '| countries:', train_set['Entity'].nunique())
print('Test set :', test_set.shape,  '| countries:', test_set['Entity'].nunique())

# Check: should print 0 (no countries in common)
print('Countries shared between train/test:', len(set(train_set['Entity']) & set(test_set['Entity'])))

## 1.5 Problem definition

**Target:** `Renewable energy share in the total final energy consumption (%)`
-- the share of renewables in **final** energy consumption (not only electricity;
it also includes traditional biomass such as firewood).

We can transform it into a **3-class classification problem**
(low / medium / high renewable share). The thresholds between classes are defined later, based
on the observed distribution in the training set.

# **2 EXPLORATORY DATA ANALYSIS**

## 2.1 Descriptive statistics and skewness

We select the "true" numerical columns (excluding `Year` and coordinates, which are
numbers but are not measurements to analyze). The key comparison in `describe()` is between
**mean** and **median (50%)**: if the mean is much larger than the median, the distribution has
a right tail. **Skewness** summarizes the asymmetry in a single number:

    0 = perfectly symmetric,
    
    >1 = right-skewed, <-1 = left-skewed.

In [ ]:
num = train_set.select_dtypes(include=np.number).drop(columns=['Year', 'Latitude', 'Longitude'])

num.describe().round(2).T[['mean', '50%', 'std', 'min', 'max']]

In [ ]:
# Sorted skewness: the most asymmetric variables are at the top
skew = num.skew().sort_values(ascending=False)
skew.round(2)

**Interpretation:** three groups. The variables in **absolute values** (TWh, CO2, flows,
density, area) have skewness values between 4 and 11: the same column contains both Kiribati and China.
They will need to be **log-transformed** if used as features. Percentage variables are
almost symmetric (bounded between 0 and 100). Access to electricity has negative skewness: a
peak at 100% and a tail towards lower values.

## 2.2 Distributions (histograms)

A histogram for each numerical variable: the fastest way to "see" all the
distributions together and identify tails, peaks, and outliers.

In [ ]:
num.hist(bins=50, figsize=(20, 16))
plt.tight_layout()
plt.show()

In [ ]:
# Focus on the effect of the logarithm: CO2 changes from strongly asymmetric to a "bell-shaped" distribution
co2 = train_set['Value_co2_emissions_kt_by_country'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].hist(co2, bins=50, color='#2f7d8c', edgecolor='white', linewidth=0.3)
axes[0].set_title(f'CO2 (kt) - natural scale · skew={co2.skew():.1f}')

axes[1].hist(np.log10(co2[co2 > 0]), bins=50, color='#2f7d8c', edgecolor='white', linewidth=0.3)
axes[1].set_title(f'log10(CO2) - after log transformation · skew={np.log10(co2[co2>0]).skew():.2f}')

plt.tight_layout(); plt.show()

## 2.3 Creation of target classes

We transform the continuous target into 3 classes (low/medium/high). The thresholds are
the empirical tertiles of the target, calculated only on the training set (to avoid
data snooping on the test set): 10.34% and 41.16%, rounded to 10 and 41. The classes
are balanced by construction, both in the training set and in the test set.

In [ ]:
# Calculate the two target tertiles, only on the training set
tertiles = train_set[target_col].quantile([0.33, 0.66])
print(tertiles)

# Visualize the target distribution with the tertile thresholds drawn above
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_set[target_col], bins=50, color='#2f7d8c', edgecolor='white', linewidth=0.3)
ax.axvline(tertiles.iloc[0], color='#d03b3b', linestyle='--')
ax.axvline(tertiles.iloc[1], color='#d03b3b', linestyle='--')
ax.set_title('Target distribution with thresholds (tertiles)')
plt.tight_layout(); plt.show()

# Round the tertiles to integers, to have more readable thresholds
low_threshold = round(tertiles.iloc[0])
high_threshold = round(tertiles.iloc[1])
print('Rounded thresholds:', low_threshold, high_threshold)

# Define the 3 intervals and their labels
thresholds = [-0.1, low_threshold, high_threshold, 100]
labels = ['low', 'medium', 'high']

# Apply the same fixed thresholds to both training and test sets
train_set['target_class'] = pd.cut(train_set[target_col], bins=thresholds, labels=labels)
test_set['target_class'] = pd.cut(test_set[target_col], bins=thresholds, labels=labels)

# Check class balance, but ONLY on the training set:
# the test set should not be inspected before the final model evaluation
print(train_set['target_class'].value_counts())

## 2.4 Correlations

We calculate the Pearson correlation coefficient between every pair of numerical columns and print only the correlations with respect to the target, in descending order.

In [ ]:
corr_matrix = train_set.corr(numeric_only=True)

corr_matrix[target_col].sort_values(ascending=False)

## 2.5 Scatter matrix on the variables most correlated with the target

The strongest correlations in absolute value are `Access to clean fuels for
cooking` (-0.79), `Access to electricity` (-0.79).

They are followed by `Low-carbon electricity`
(0.46), `Primary energy consumption per capita (kWh/person)` (-0.42), and `gdp_per_capita` (-0.37).

In [ ]:
# Scatterplot of correlations > 0.5 (absolute value)

from pandas.plotting import scatter_matrix

# Select the target and the 3 variables most correlated with it
attributes = [
    target_col,
    "Access to clean fuels for cooking",
    "Access to electricity (% of population)"
]

# Short labels to use instead of the long column names
short_labels = ["RENEWABLE SHARE (Target) [%]", "Access to clean fuels", "Access to electricity [% pop.]"]

# Draw the scatterplot matrix (and histograms on the diagonal)
axes = scatter_matrix(train_set[attributes], figsize=(14, 11))

# Adjust axis labels: only first column (Y) and last row (X)
for i in range(len(attributes)):
    axes[i, 0].set_ylabel(short_labels[i], fontsize=8)
    axes[-1, i].set_xlabel(short_labels[i], fontsize=8)

    # Reduce axis number size and rotate labels to avoid overlap
    for ax in axes[i, :]:
        ax.tick_params(axis='both', labelsize=6)
        for label in ax.get_xticklabels():
            label.set_rotation(45)

plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot of correlations < 0.5 (absolute value)

from pandas.plotting import scatter_matrix

# Select the target and the 3 variables most correlated with it
attributes = [
    target_col,
    "gdp_per_capita",
    "Primary energy consumption per capita (kWh/person)",
    "Low-carbon electricity (% electricity)"
]

# Short labels to use instead of the long column names
short_labels = ["RENEWABLE SHARE (Target) [%]", "GDP per capita", "Energy consumption [kWh/person]", "Low-carbon electr. [%]"]

# Draw the scatterplot matrix (and histograms on the diagonal)
axes = scatter_matrix(train_set[attributes], figsize=(14, 11))

# Adjust axis labels: only first column (Y) and last row (X)
for i in range(len(attributes)):
    axes[i, 0].set_ylabel(short_labels[i], fontsize=8)
    axes[-1, i].set_xlabel(short_labels[i], fontsize=8)

    # Reduce axis number size and rotate labels to avoid overlap
    for ax in axes[i, :]:
        ax.tick_params(axis='both', labelsize=6)
        for label in ax.get_xticklabels():
            label.set_rotation(45)

plt.tight_layout()
plt.show()

**Scatterplot observations:**

1. **Accumulation at value 100.** In the plots involving `Access electricity` and
`Clean fuels` (especially their cross relationship), a dense band of points can be seen
right on the 100 line: many countries now have almost complete access to electricity
and clean fuels, so they are concentrated at the maximum possible percentage value.

2. **Diagonal trails.** In several plots (e.g. Target vs `Access to clean fuels`)
small diagonal traces can be observed instead of a homogeneous cloud: these are the
same countries appearing multiple times (one row per year) and gradually changing
over time, leaving a "trail" of nearby points.

3. **`Access to clean fuels` and `Access to electricity` are almost redundant
with each other** (correlation 0.87, even stronger than their correlation with the target):
they both measure the country's level of infrastructural development.
`Low-carbon electricity`, on the other hand, is almost unrelated to the other two
(correlation around -0.13/-0.15): it seems to depend more on energy policy choices
than on the level of economic development.

4. **Weak but non-random correlations.** `gdp_per_capita` and `Energy
consumption per capita` remain concentrated close to zero, with a few countries
deviating significantly (consistent with the skewness observed in 2.1): against
the target they produce sparse clouds, without a clear trend. Between them, however,
a clearer increasing pattern can be observed: richer countries consume more energy
per capita, a relationship stronger than the one both variables have with the target.

## 2.6 Visualizing geographic data

In [ ]:
# First scatter: only the geographic position of each country-year
train_set.plot(kind="scatter", x="Longitude", y="Latitude", grid=True, figsize=(10,7))
plt.show()

# Create explicit axes, which will be needed to add labels later
fig, ax = plt.subplots(figsize=(10, 7))

# Same scatter, but color each point based on the target value
train_set.plot(kind="scatter", x="Longitude", y="Latitude", grid=True,
                c=target_col, cmap="jet", colorbar=True,
                legend=True, sharex=False, ax=ax)

# One reference country per continent, for map orientation
reference_countries = ['Italy', 'United States', 'Brazil', 'Nigeria', 'China', 'Australia']

# Add the name of each country near its point
for country in reference_countries:
    sub = train_set[train_set['Entity'] == country]
    if sub.empty:
        continue                       # country not in train (it is in test): skip it
    row = sub.iloc[0]
    ax.annotate(country, (row['Longitude'], row['Latitude']),
                fontsize=9, fontweight='bold',
                xytext=(5, 5), textcoords='offset points')

plt.show()

The colored map reveals a pattern that may seem counterintuitive: countries such as
Nigeria show a high renewable energy share, higher than many European countries.
This depends on the definition of the target, which includes traditional biomass
(firewood): in poorer countries, where people still cook with firewood due to a lack
of alternatives, the target appears "high" not because of a modern energy transition,
but because of **energy poverty**. This is a known limitation of this metric
(documented also by IEA and Our World in Data), which must be taken into account when
interpreting the model results.

## 2.7 Feature engineering

### 2.7.1 **Estimated population**

Density (inhabitants/Km²) multiplied by area (Km²) gives an estimate of the
country's total population in that year. We calculate it separately on
`train_set` and `test_set`, because it is a row-by-row transformation (it does not require
statistics calculated on the entire dataset, so there is no risk of data
snooping).

In [ ]:
# Estimated population = population density × country area
train_set['estimated_population'] = train_set['Density\\n(P/Km2)'] * train_set['Land Area(Km2)']
test_set['estimated_population'] = test_set['Density\\n(P/Km2)'] * test_set['Land Area(Km2)']

# Quick check: compare with some known countries to verify that the values
# are reasonable
top10 = train_set[['Entity', 'Year', 'estimated_population']].sort_values('estimated_population', ascending=False).head(10)
top10['estimated_population'] = top10['estimated_population'].map('{:,.0f}'.format)
print(top10)

### 2.7.2 **CO2 per capita**

We divide total CO2 emissions (kt) by the estimated population, in order
to obtain an indicator that can be compared across countries of different sizes.

In [ ]:
train_set['co2_per_capita'] = train_set['Value_co2_emissions_kt_by_country'] / train_set['estimated_population']
test_set['co2_per_capita'] = test_set['Value_co2_emissions_kt_by_country'] / test_set['estimated_population']

print(train_set['co2_per_capita'].describe())

**Note:** 2553 valid rows out of 2764. The mean (0.0046 kt/person ≈ 4.6 tonnes/person) remains
consistent with the real global average, but it is higher than the median (0.0026). There is
also a right tail here: a few countries (large oil/gas producers with small populations)
have very high per-capita emissions (up to about 47 tonnes), while the majority of countries
remain at low values. As with raw CO2 emissions, it should be log-transformed if used as a
feature in a model.

### 2.7.3 **People without access to electricity**

We transform the percentage of the population **without** access to electricity
(100 - Access to electricity) into an absolute number of people, multiplying it
by the estimated population. We obtain an indicator that can be compared across
countries of different sizes.

In [ ]:
train_set['people_without_electricity'] = train_set['estimated_population'] * (100 - train_set['Access to electricity (% of population)']) / 100
test_set['people_without_electricity'] = test_set['estimated_population'] * (100 - test_set['Access to electricity (% of population)']) / 100

print(train_set['people_without_electricity'].describe().map('{:,.0f}'.format))

# The 10 country-year observations with the highest number of people without access to electricity
top10 = train_set[['Entity', 'Year', 'people_without_electricity']].sort_values('people_without_electricity', ascending=False).head(10)
top10['people_without_electricity'] = top10['people_without_electricity'].map('{:,.0f}'.format)
print(top10)

## 2.8 EDA control questions

**1. Which variable seems to be most related to the renewable energy share
(the target)?**

The two strongest ones are `Access to clean fuels for cooking` (-0.79) and `Access to
electricity` (-0.79), followed by `Low-carbon electricity` (0.46) with a positive
correlation.
The first two are negatively correlated: a more developed country tends to
have a lower renewable share according to this target, due to the traditional
biomass included in the definition. Therefore, infrastructure development
variables are the ones most associated with the target, but the direction of the
relationship must be explained; it is not simply "more development = more renewables"
as one might intuitively expect.

**2. Are there strange values or artificial limits in the data?**

Yes, several:
- accumulation of points at the maximum value (100%) for `Access to electricity` and
`Access to clean fuels`: many countries have already reached saturation in that percentage;
- the target includes traditional biomass, causing the paradox of countries such as
Nigeria appearing more "renewable" than many European countries;
- an anomalous value in the external density dataset (Sudan, 2000) was
identified and treated as missing;

**3. Does geography seem important?**

Yes, partially: `Latitude` has a correlation of -0.32 with the target (not
very strong, but not negligible), and the colored map shows visible regional
patterns. It is not the strongest factor overall, but it represents a real signal,
not noise.

**4. Do the features created through ratios/combinations of columns seem more
informative than the original columns?**

Yes:
- `co2_per_capita` is more informative than absolute CO2 emissions when comparing countries
of different sizes, although it still inherits the asymmetry (skewness) of the original CO2 variable;
- `people_without_electricity` adds real-scale context (number of people, not just percentage)
compared to the original column, and being based on corrected density, it correctly reflects
demographic evolution over time.

In general, the created features are more interpretable and useful both for
comparisons between countries and for changes over time, after the population
density correction performed in section 1.1.

# 3 PREPROCESSING

In [ ]:
# Returns the number of NaN values for each column, sorted from worst to best
df.isna().sum().sort_values(ascending=False)

In [ ]:
# Removal of rows related to French Guiana
df = df[df['Entity'] != 'French Guiana'].reset_index(drop=True)

# Immediate verification of the successful removal
print("Remaining rows for French Guiana:", len(df[df['Entity'] == 'French Guiana']))

**Feature selection.** We exclude three groups of columns: identifiers/labels
(`Entity`, the continuous target, `target_class`); **leakage variables** — the target
in another form: `Electricity from renewables/fossil/nuclear (TWh)`,
`Low-carbon electricity (%)`, `Renewable-electricity-generating-capacity-per-capita`;
and columns **redundant** with the engineered features from section 2.7
(`Value_co2...` → `co2_per_capita`,
`Land Area` → `estimated_population`, `Longitude`, with correlation ~0 with the target).
The result is **12 informative and non-leaking features**.

Two transformation branches: skewed variables (median imputation → `log1p` →
standardization) and linear variables (median imputation → standardization).

### Why we exclude leakage variables (demonstration)

The absolute TWh values taken individually have almost zero correlation with the target (they are misleading),
but their **ratio** reconstructs the renewable share: this is why they must be removed.

In [ ]:
# DEMONSTRATION of hidden leakage: individual TWh values seem unrelated to the target,
# but their RATIO reconstructs the renewable electricity share.

total = (train_set['Electricity from fossil fuels (TWh)']
         + train_set['Electricity from nuclear (TWh)']
         + train_set['Electricity from renewables (TWh)'])

derived_share = train_set['Electricity from renewables (TWh)'] / total.replace(0, np.nan) * 100

print('r  Electricity from renewables (TWh) alone :',
      round(train_set['Electricity from renewables (TWh)'].corr(train_set[target_col]), 3))

print('r  Low-carbon electricity (%)                :',
      round(train_set['Low-carbon electricity (% electricity)'].corr(train_set[target_col]), 3))

print('r  share DERIVED from the TWh ratio          :',
      round(derived_share.corr(train_set[target_col]), 3))

### 3.1 Feature selection and definition of X / y

In [ ]:
# Cleaning: rename the density column to "Density"
density_col = [c for c in train_set.columns if c.startswith('Density')][0]
train_set = train_set.rename(columns={density_col: 'Density'})
test_set  = test_set.rename(columns={density_col: 'Density'})

# Labels (y): the target class created in section 2.3
y_train = train_set['target_class']
y_test  = test_set['target_class']

# Features to be LOG-transformed: highly asymmetric positive values (high skew, 2.1).
# log1p = log(1+x): compresses long tails and handles zeros without producing -inf.
log_feat = [
    'Primary energy consumption per capita (kWh/person)',
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)',
    'gdp_per_capita',
    'Density',
    'estimated_population',        # engineered 2.7.1
    'co2_per_capita',             # engineered 2.7.2 (replaces raw CO2)
    'people_without_electricity',  # engineered 2.7.3
]

# LINEAR features: percentages, rates and coordinates, already on a good scale (no log).
lin_feat = [
    'Access to electricity (% of population)',
    'Access to clean fuels for cooking',
    'gdp_growth',
    'Latitude',
    'Year',
]


# EXCLUDED intentionally:
#   - identifiers/labels: Entity, continuous target, target_class
#   - LEAKAGE (masked target): Electricity from renewables/fossil/nuclear (TWh),
#     Low-carbon electricity (%), Renewable-electricity-generating-capacity-per-capita
#   - redundant with engineered features: Value_co2_emissions_kt (-> co2_pro_capite),
#     Land Area(Km2) (-> popolazione_stimata), Longitude (corr. ~0 with target)

features = log_feat + lin_feat
X_train = train_set[features]
X_test  = test_set[features]

print('Number of features:', len(features))
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

### 3.2 The preprocessing pipeline (ColumnTransformer)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

# LOG branch: median imputation -> log1p (compresses long tails) -> standardization
log_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log',     FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler',  StandardScaler()),
])

# LINEAR branch: median imputation -> standardization
lin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# The ColumnTransformer applies each branch to its columns and recombines them
preprocessing = ColumnTransformer([
    ('log', log_pipe, log_feat),
    ('lin', lin_pipe, lin_feat),
])

preprocessing

In [ ]:
# 1. Apply preprocessing and transform the data
X_train_preprocessed = preprocessing.fit_transform(X_train)

# 2. Check for the presence of NaN values
# If the output is a NumPy matrix:
nan_count = np.isnan(X_train_preprocessed).sum()

print("--- MISSING VALUES CHECK (NaN) AFTER PREPROCESSING ---")
print(f"Total number of NaN remaining in the transformed dataset: {nan_count}")

if nan_count == 0:
    print("Confirmed: all NaNs have been successfully removed through median imputation.")
else:
    print("Warning: missing values are still present.")

# Convert the output into a DataFrame using the feature names tracked by the transformer
X_train_df = pd.DataFrame(
    X_train_preprocessed,
    columns=preprocessing.get_feature_names_out()
)

# Display the NaN count for each transformed column
print(X_train_df.isna().sum())

# 4 CROSS-VALIDATION AND TUNING

We compare 3 algorithms:

1. **Logistic Regression** — linear baseline
2. **Decision Tree Classifier** — non-linear model
3. **Random Forest Classifier** — tree ensemble

Each model is included in a `make_pipeline(preprocessing, model)`. We evaluate using
**`StratifiedGroupKFold` (cv=3)**: it combines two requirements — **groups** (no country is split between
folds, consistent with the country-based split from section 1.4) and **stratification** (class
proportions remain constant).

This way, cross-validation also measures generalization to new countries, without panel
leakage.

Metrics: **accuracy** and **macro F1**.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# groups = the country of each row in the training set (required for grouped CV)
groups_train = train_set['Entity']
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42, n_jobs=-1),
}

results = []

for name, model in models.items():
    pipe = make_pipeline(preprocessing, model)   # preprocessing + model together

    acc = cross_val_score(pipe, X_train, y_train, groups=groups_train,
                          scoring='accuracy', cv=cv, n_jobs=-1)

    f1 = cross_val_score(pipe, X_train, y_train, groups=groups_train,
                         scoring='f1_macro', cv=cv, n_jobs=-1)

    results.append({
        'Model': name,
        'Accuracy CV (mean)': acc.mean(),
        'Accuracy CV (std)': acc.std(),
        'Macro F1 CV (mean)': f1.mean(),
        'Macro F1 CV (std)': f1.std()
    })

pd.DataFrame(results).sort_values('Macro F1 CV (mean)', ascending=False).reset_index(drop=True).round(3)

In [ ]:
# Isolation of the first validation split
for train_idx, val_idx in cv.split(X_train, y_train, groups=groups_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    entities_val = groups_train.iloc[val_idx]
    break

# Dictionary to accumulate table data
comparison_table = {
    'Country': entities_val,
    'Actual Value': y_val
}

# Training and collection of predictions in the required order
for name in ['Random Forest', 'Logistic Regression', 'Decision Tree']:
    pipe = make_pipeline(preprocessing, models[name])
    pipe.fit(X_tr, y_tr)
    comparison_table[name] = pipe.predict(X_val)

# Creation of the DataFrame and display of an exact sample of 10 rows
df_examples = pd.DataFrame(comparison_table)
df_examples.sample(n=10).reset_index(drop=True)

**1. Which model performs best on the training set?**

Random Forest followed by Logistic Regression.

**2. Which model seems to generalize better?**

Random Forest, as it records a significantly lower validation error.

**3. Why is cross-validation more reliable?**

Because it evaluates the model on data not seen during training, simulating production behavior and eliminating the illusion of accuracy caused by simply memorizing the training set.

**5. Why does increasing flexibility worsen generalization?**

A model with too much flexibility (e.g. overly deep trees) captures statistical noise and random fluctuations in the training set instead of learning the general rule, failing as soon as the data changes slightly.

## 4.1 Logistic Regression: the weight of each variable for each class (coefficients)

In [ ]:
pipe_lr = make_pipeline(preprocessing, LogisticRegression(max_iter=1000, random_state=42))
pipe_lr.fit(X_train, y_train)

lr = pipe_lr.named_steps['logisticregression']
coef_df = pd.DataFrame(lr.coef_, columns=features, index=lr.classes_)

fig, ax = plt.subplots(figsize=(10, 5))
coef_df.T.plot(kind='barh', ax=ax)

ax.set_xlabel('Coefficient')
ax.set_title('Coefficients - Logistic Regression (per class)')

plt.tight_layout()
plt.show()

`co2_per_capita` is the variable with the strongest coefficient (±3.3), followed
by `gdp_per_capita` (±1.7) and `Access to clean fuels for cooking` (±1.4): they
are "axes" that clearly separate the low and high classes, while having little
impact on the medium class. `Year` and `gdp_growth` are almost irrelevant.

## 4.2 Decision Tree: the decision thresholds learned by the model

In [ ]:
# Train the Decision Tree on the entire training set (not just one CV fold)
# to visualize the tree that the model actually learns.

from sklearn.tree import plot_tree

# Preprocessing + Decision Tree pipeline, trained on the full training set
pipe_tree = make_pipeline(preprocessing, DecisionTreeClassifier(random_state=42))
pipe_tree.fit(X_train, y_train)

tree_model = pipe_tree.named_steps['decisiontreeclassifier']

print('Tree depth:', tree_model.get_depth())
print('Number of leaves:', tree_model.get_n_leaves())

In [ ]:
plt.figure(figsize=(28, 12))  # much wider, less tall

plot_tree(tree_model,
          feature_names=features,
          class_names=sorted(y_train.unique()),
          filled=True,
          max_depth=2,        # try 2 levels instead of 3, more readable
          fontsize=9)

plt.title("Decision Tree (first 2 levels)")
plt.show()

**Observations on the Decision Tree:**

The root node (the first decision, and the most important one for the model) is
`co2_per_capita <= -0.562`, not one of the original variables: this confirms that
the previously engineered feature provides useful information, enough to become
the first splitting criterion.

Interesting is the contrast with the EDA: the strongest linear correlations with
the target were `Access to clean fuels` and `Access to electricity` (-0.79), but
the tree does not use them in the first 2 levels — it prefers `co2_per_capita`,
`Energy intensity`, and `Primary energy consumption per capita`. An
tree-based model can capture non-linear relationships that a simple Pearson
correlation cannot detect.

## 4.3 Random Forest: which variables matter most (feature importance)

In [ ]:
pipe_rf = make_pipeline(preprocessing, RandomForestClassifier(random_state=42, n_jobs=-1))
pipe_rf.fit(X_train, y_train)

rf = pipe_rf.named_steps['randomforestclassifier']

importances = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))

importances.plot(kind='barh', ax=ax, color='#2f7d8c')

ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature importance - Random Forest')

plt.tight_layout()
plt.show()

For the Random Forest, `co2_per_capita` is also the most important variable
(0.178), followed by `Primary energy consumption per capita` (0.151) and
`Latitude` (0.121). This is consistent with the Logistic Regression results for
`co2_per_capita` and `Access to clean fuels for cooking`, but `Latitude` emerges
as more relevant here than in the linear coefficients: the Random Forest
probably captures a non-linear geographical effect that the linear model cannot
detect.

`Year` and `gdp_growth` remain the least informative variables in both models.

(Mean decrease in impurity)

# 5 FINAL TEST

### **5.1 Analisi dei risultati sul Test Set**

In [ ]:
from sklearn.metrics import classification_report

# Final training on the entire training set and prediction on the test set
pipe_rf.fit(X_train, y_train)

y_pred = pipe_rf.predict(X_test)
y_score = pipe_rf.predict_proba(X_test)

# Extraction of class labels directly from the fitted model
class_labels = [str(c) for c in pipe_rf.classes_]

# Print aggregate metrics
print("--- CLASSIFICATION REPORT (TEST SET) ---")
print(classification_report(y_test, y_pred, target_names=class_labels))

The Random Forest model shows a solid ability to generalize to completely unseen countries during the training phase, achieving an overall **accuracy of 69%** and a **macro F1-score of 0.70**. This behavior confirms the effectiveness of the group-based split (*GroupShuffleSplit*) in avoiding temporal or geographical data leakage phenomena.

#### **Class-level details:**

*   **"Low" class:** Achieves the highest performance, with a *precision* of **0.81** and an *F1-score* of **0.79**. The model produces few false positives in this range, accurately identifying contexts with a low share of renewable energy.
*   **"Medium" class:** Shows the highest *recall* value (**0.81**), indicating that almost all countries belonging to this intermediate range are detected, despite a lower *precision* (**0.63**) due to incorrect inclusions from the other classes.
*   **"High" class:** Represents the most difficult class to discriminate, with a *recall* dropping to **0.56** and an *F1-score* of **0.60**.

#### **Analytical considerations:**

The decline observed in the "high" class directly reflects the structural limitations analyzed during the EDA phase. The inclusion of traditional biomass (e.g., firewood) in the overall target calculation introduces a strong distortion between countries with low electrification (energy poverty) and countries with an advanced modern ecological transition. Tree-based models struggle to define a clear boundary for this class because the target groups under the same label macroeconomic and infrastructural contexts that are completely different.

### **5.2 Confusion Matrix Analysis**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))

cm = confusion_matrix(y_test, y_pred, labels=class_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_labels
)

disp.plot(cmap='Blues', ax=ax, values_format='d')

ax.set_title('Confusion Matrix - Test Set')

plt.tight_layout()
plt.show()

**Absence of bipolar errors:** The model never makes the mistake of confusing the "low" class with the "high" class or vice versa (the values in the opposite corners of the matrix are exactly 0). This indicates that the separation between the two extremes of the energy spectrum is clear and without ambiguity.

**Balance of correct predictions:** The number of true positives remains stable and balanced across all three categories (158 for low, 162 for medium, 159 for high), confirming that the classifier does not show evident bias toward a specific class.

### **5.3 Multiclass ROC Curve Analysis (One-vs-Rest)**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import roc_curve, auc

lb = LabelBinarizer()
lb.fit(y_train)

y_test_bin = lb.transform(y_test)
n_classes = len(class_labels)

fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#d03b3b', '#2f7d8c', '#e6a15c']

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)

    ax.plot(
        fpr,
        tpr,
        color=colors[i],
        lw=2,
        label=f'Class {class_labels[i]} (AUC = {roc_auc:.2f})'
    )

ax.plot([0, 1], [0, 1], 'k--', lw=1.5)

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('Multiclass ROC Curves - Random Forest (Test Set)')

ax.legend(loc="lower right")
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

#### **Class performance details:**

**"Low" class (AUC = 0.97):** Shows an almost perfect separation capability compared with the other classes. The curve quickly approaches the ideal point (0,1), indicating that the model can discriminate extremely effectively between countries with a low share of renewable energy.

**"Medium" class (AUC = 0.93):** Despite the lower precision highlighted in previous reports, the area under the curve remains at an excellent level. This demonstrates that the model's intrinsic probabilistic ranking ability for the intermediate category is still very strong.

**"High" class (AUC = 0.81):** This confirms the most difficult class to model. Although the value is clearly above a random classifier (represented by the dashed diagonal line with AUC = 0.50), the curve has a more moderate slope, caused by the structural ambiguities of the target metric already discussed during the EDA phase.

#### **Analytical considerations:**

The combined analysis of the three AUC values highlights that the algorithm has a strong ability to distinguish broad differences in the energy mix among countries. The model is particularly reliable for classification tasks aimed at identifying situations with low renewable energy penetration, while greater caution or the integration of additional macroeconomic features is required for the finer categorization of contexts with high renewable shares.

# **6 CONCLUSIONS AND FUTURE DEVELOPMENTS**

### **Summary of the work performed**

The objective of classifying countries according to their share of renewable energy in final energy consumption (low, medium, and high categories) was achieved by implementing a rigorous and reproducible workflow:

1. **Data Cleaning & Integration:**  
   The population density variable was corrected through the integration of an external dataset, resolving the structural anomalies identified during the preliminary analysis.

2. **Feature Engineering:**  
   The creation of per-capita indicators (such as `co2_per_capita` and `people_without_electricity`) allowed the normalization of demographic and geographical differences among countries, providing some of the most significant predictors for tree-based models.

3. **Robust Validation:**  
   The split and cross-validation procedures were performed strictly by groups (`Entity`), ensuring a realistic evaluation of generalization capability and eliminating the risk of spatial and temporal data leakage.

4. **Modeling:**  
   The comparison identified **Random Forest** as the optimal architecture, thanks to its ability to capture non-linear relationships within the data, achieving an accuracy of **69%** and a macro F1-score of **0.70** on the test set.

### **Limitations and future developments**

The analysis of error metrics and ROC curves confirmed that the "high" class is affected by an intrinsic limitation of the target variable. The inclusion of traditional biomass creates confusion between contexts of energy poverty and those characterized by a genuine advanced ecological transition.

A possible future development would be to disaggregate the target variable by separating modern renewable sources (wind, solar, hydroelectric) from traditional biomass, or to integrate additional indicators related to electricity grid efficiency in order to improve the model's accuracy for countries with the highest renewable energy penetration.